# Simple Linear Regression — Complete Visual Learning Notebook

**From intuition to implementation, interpretation, evaluation, and limitations**

This notebook is designed as a teaching resource. It explains each idea in plain language, uses diagrams and plots, and builds a complete model step by step.

## Learning objectives

By the end, you should be able to:
1. Explain what regression is and when to use it.
2. Understand features, targets, slope, intercept, predictions, and residuals.
3. Read scatter plots and recognize positive, negative, and no linear relationship.
4. Fit a simple linear regression model with scikit-learn.
5. Interpret the fitted equation.
6. Evaluate predictions using MAE, MSE, RMSE, and R².
7. Understand assumptions, common mistakes, and practical limitations.

> **Important:** The car dataset below is illustrative and generated for learning. It is not real market data and should not be used for actual vehicle pricing.


## 1. What is regression?

Regression is a supervised machine-learning method used to predict a **numeric value**.

Examples:
- Predict house price from area.
- Predict salary from years of experience.
- Predict a car's price from engine displacement.
- Predict electricity consumption from temperature.

In supervised learning, the model learns from examples where both the input and the correct output are known.

### Simple linear regression

Simple linear regression uses **one input feature** to predict one numeric target. It tries to describe the relationship with a straight line:

\[
\hat{Y} = mX + c
\]

| Symbol | Meaning |
|---|---|
| \(X\) | Input feature (independent variable) |
| \(\hat{Y}\) | Predicted output |
| \(m\) | Slope / coefficient |
| \(c\) | Intercept |
| \(Y\) | Actual observed output |

**Intuition:** The model finds the straight line that stays as close as possible to the data points overall.


## 2. A visual overview of the machine-learning workflow

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make plots look clean and readable
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

fig, ax = plt.subplots(figsize=(12, 2.8))
steps = ["Collect data", "Explore data", "Split data", "Train model", "Predict", "Evaluate"]
xs = np.linspace(0.08, 0.92, len(steps))
for i, (x, label) in enumerate(zip(xs, steps)):
    ax.text(x, 0.5, label, ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.6", fc="white", ec="black"))
    if i < len(steps)-1:
        ax.annotate("", xy=(xs[i+1]-0.07, 0.5), xytext=(x+0.07, 0.5),
                    arrowprops=dict(arrowstyle="->", lw=1.5))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.set_title("Typical supervised-learning workflow", pad=12)
plt.show()

## 3. Create a small car dataset

We will predict a car's **Price** from its engine displacement (**Disp.**).

- `Disp.` is the feature, often represented as X.
- `Price` is the target, often represented as Y.

The values below are synthetic and intentionally contain some variation so the model has something realistic to learn.


In [ ]:
rng = np.random.default_rng(42)

# Synthetic engine displacement values
disp = np.array([1.0, 1.2, 1.4, 1.5, 1.6, 1.8, 2.0, 2.2, 2.4, 2.5,
                 2.7, 3.0, 3.2, 3.5, 3.8, 4.0, 4.2, 4.5, 4.8, 5.0])

# Illustrative price relationship plus random variation
price = 4.5 + 2.2 * disp + rng.normal(0, 0.75, size=len(disp))

cars = pd.DataFrame({"Disp.": disp, "Price": price})
cars.head()

In [ ]:
print("Dataset shape:", cars.shape)
print("\\nColumn information:")
print(cars.info())
print("\\nSummary statistics:")
display(cars.describe())

### Understand the data

A row is one example (one car). A column is one variable.

Before training, check:
- How many rows and columns are present?
- Are there missing values?
- Are the values numeric?
- Are there unusual values or outliers?


In [ ]:
print("Missing values per column:")
display(cars.isna().sum())

print("Duplicate rows:", cars.duplicated().sum())

print("Data types:")
display(cars.dtypes)

## 4. Explore the relationship with a scatter plot

A scatter plot places each car as a dot:
- Horizontal axis: engine displacement.
- Vertical axis: price.

Look for the overall pattern. Do prices tend to rise as displacement rises? Is the relationship roughly straight, or does it curve?


In [ ]:
plt.figure()
plt.scatter(cars["Disp."], cars["Price"], s=65)
plt.xlabel("Engine displacement (Disp.)")
plt.ylabel("Car price (illustrative units)")
plt.title("Engine displacement vs car price")
plt.show()

## 5. Understand the direction of slope

The slope tells us the direction and rate of change in the predicted output as X increases.

- **Positive slope:** predicted Y increases as X increases.
- **Negative slope:** predicted Y decreases as X increases.
- **Zero slope:** the line is flat; predicted Y does not change with X.

The following diagrams are conceptual examples, not fitted models.


In [ ]:
x_demo = np.linspace(0, 10, 100)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

examples = [
    (2*x_demo + 1, "Positive slope", "As X increases, Y increases"),
    (-1.2*x_demo + 12, "Negative slope", "As X increases, Y decreases"),
    (np.full_like(x_demo, 5), "Zero slope", "Predicted Y stays constant")
]
for ax, (y_demo, title, subtitle) in zip(axes, examples):
    ax.plot(x_demo, y_demo, linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Predicted Y")
    ax.text(0.5, -0.24, subtitle, transform=ax.transAxes, ha="center", wrap=True)
plt.tight_layout()
plt.show()

## 6. The line equation: slope and intercept

\[
\hat{Y} = mX + c
\]

### Slope (m)
The slope is the predicted change in Y for a one-unit increase in X.

Example: if \(m=2.2\), then a one-unit increase in displacement is associated with a predicted increase of 2.2 price units.

### Intercept (c)
The intercept is the predicted value of Y when X equals zero.

The intercept is needed to position the line. It may not have a useful real-world interpretation if X = 0 is outside the meaningful range of the data.


In [ ]:
# A simple numerical example
m = 2.2
c = 4.5
x_value = 3.0
predicted_y = m * x_value + c

print(f"Equation: Ŷ = {m}X + {c}")
print(f"When X = {x_value}, predicted Y = {predicted_y:.2f}")

## 7. Split the data into training and testing sets

Why split?
- **Training data** teaches the model.
- **Testing data** checks how well it predicts examples it did not train on.

If we evaluate only on training data, the score can look overly optimistic. The test set gives a more realistic check of performance on unseen data.

For a small dataset, test scores can vary noticeably depending on the split. In real projects, use more data and consider cross-validation.


In [ ]:
X = cars[["Disp."]]   # Keep X as a 2D table for scikit-learn
y = cars["Price"]      # y is the target series

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))

## 8. Train the simple linear regression model

`LinearRegression()` creates the model. `fit()` makes it learn the best-fitting line from the training examples.

In ordinary least squares regression, the model chooses the line that minimizes the **sum of squared residuals** on the training data.


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

slope = model.coef_[0]
intercept = model.intercept_

print("Slope (coefficient):", slope)
print("Intercept:", intercept)
print(f"Learned equation: Ŷ = {slope:.3f} × Disp. + {intercept:.3f}")

## 9. Visualize the fitted regression line

The dots are the observed examples. The line is the model's prediction across the range of displacement values.

The line does not need to pass through every dot. Real data often contains noise and influences that are not included in the model.


In [ ]:
x_line = np.linspace(cars["Disp."].min(), cars["Disp."].max(), 100)
y_line = model.predict(pd.DataFrame({"Disp.": x_line}))

plt.figure()
plt.scatter(X_train["Disp."], y_train, label="Training data")
plt.scatter(X_test["Disp."], y_test, label="Testing data", marker="s")
plt.plot(x_line, y_line, linewidth=2.5, label="Fitted regression line")
plt.xlabel("Engine displacement (Disp.)")
plt.ylabel("Car price (illustrative units)")
plt.title("Simple linear regression fit")
plt.legend()
plt.show()

## 10. Make predictions

Use `predict()` to estimate the target for input values.

A prediction is an estimate, not a guarantee. It is most trustworthy when the new input is within the range represented in the training data.


In [ ]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    "Actual price": y_test.values,
    "Predicted price": y_pred
})
results["Residual"] = results["Actual price"] - results["Predicted price"]
display(results.round(3))

## 11. Residuals: how far off were the predictions?

A **residual** is:

\[
e = Y - \hat{Y}
\]

- Positive residual: actual value is higher than predicted.
- Negative residual: actual value is lower than predicted.
- Residual near zero: prediction was close.

The residual is a signed error. Most evaluation metrics summarize the size of these errors across many examples.


In [ ]:
plt.figure()
plt.scatter(y_pred, y_test - y_pred)
plt.axhline(0, linestyle="--", linewidth=2)
plt.xlabel("Predicted price")
plt.ylabel("Residual (actual − predicted)")
plt.title("Residual plot")
plt.show()

## 12. Evaluation metrics

No single metric tells the whole story. Read the metrics together and consider the scale and context of the target.

### MAE — Mean Absolute Error

\[
MAE = \frac{1}{n}\sum_{i=1}^{n}|Y_i-\hat{Y}_i|
\]

Average absolute prediction error. It is in the same units as the target and is relatively easy to explain.

### MSE — Mean Squared Error

\[
MSE = \frac{1}{n}\sum_{i=1}^{n}(Y_i-\hat{Y}_i)^2
\]

Squares errors, so large mistakes receive much more weight. Its units are squared.

### RMSE — Root Mean Squared Error

\[
RMSE = \sqrt{MSE}
\]

Like MAE, RMSE is in the target's units. It penalizes large errors more strongly than MAE.

### R² — Coefficient of determination

\[
R^2 = 1 - \frac{\sum (Y_i-\hat{Y}_i)^2}{\sum (Y_i-\bar{Y})^2}
\]

It compares the model with a baseline that always predicts the mean of the observed target values.
- 1.0: perfect predictions on this evaluated dataset.
- 0.0: no better than predicting the test-set mean, under the standard definition.
- Below 0: worse than that mean-prediction baseline.

R² is **not** the percentage of predictions that are correct, and a high R² does not prove causation.


In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R²"],
    "Value": [mae, mse, rmse, r2],
    "What it tells us": [
        "Average absolute error, in target units",
        "Average squared error; emphasizes large misses",
        "Typical error scale, in target units; emphasizes large misses",
        "Performance relative to predicting the test-set mean"
    ]
})
display(metrics.round(4))

### How should you interpret the values?

Suppose the MAE is 0.6 price units. A simple explanation is: “On average, the model's predictions are about 0.6 price units away from the actual values.”

Whether 0.6 is good depends on the scale and use case. An error of 0.6 may be small for a very expensive item but large for a low-cost item.

Do not compare MAE or RMSE across datasets with very different target scales without considering that difference.


## 13. Compare predictions with actual values

A useful diagnostic is to plot actual values against predicted values. If predictions were perfect, every point would fall on the diagonal line.


In [ ]:
plt.figure()
plt.scatter(y_test, y_pred, s=70)
low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())
plt.plot([low, high], [low, high], linestyle="--", linewidth=2, label="Perfect prediction")
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.title("Actual vs predicted values")
plt.legend()
plt.show()

## 14. Predict for a new example

Provide a new displacement value as a one-column DataFrame. The column name must match the feature used during training.


In [ ]:
new_car = pd.DataFrame({"Disp.": [2.8]})
new_price = model.predict(new_car)[0]

print(f"Predicted illustrative price for Disp. = 2.8: {new_price:.2f}")

## 15. What does “best-fit line” mean?

For each training point, the vertical distance to the line is its residual. Ordinary least squares chooses the line that minimizes the sum of squared vertical residuals.

This is why the method is called **least squares**.


In [ ]:
# Visualize residual distances for a few training observations
sample = X_train.iloc[:8].copy()
sample_actual = y_train.loc[sample.index]
sample_pred = model.predict(sample)

plt.figure()
plt.scatter(X_train["Disp."], y_train, label="Training observations")
plt.plot(x_line, y_line, label="Regression line")
for x, actual, pred in zip(sample["Disp."], sample_actual, sample_pred):
    plt.plot([x, x], [actual, pred], linestyle=":", linewidth=1.5)
plt.xlabel("Engine displacement")
plt.ylabel("Price")
plt.title("Vertical residual distances")
plt.legend()
plt.show()

## 16. Assumptions and checks

Simple linear regression is most useful when a straight-line relationship is a reasonable approximation.

Common assumptions / considerations:

1. **Linearity:** the mean relationship between X and Y is approximately linear.
2. **Independent errors:** observations/errors are not dependent in a way the model ignores.
3. **Constant error variance (homoscedasticity):** residual spread is reasonably consistent across fitted values.
4. **Residual distribution:** normality is mainly relevant for some statistical inference and confidence intervals, not a requirement for computing predictions.
5. **No extreme influential points:** outliers can strongly affect the fitted line.
6. **Appropriate measurement and data quality:** missing values, errors, and biased sampling can undermine results.

A residual plot can reveal curvature or changing spread. It cannot prove that every assumption is satisfied.


In [ ]:
# Residuals against the feature
train_pred = model.predict(X_train)
train_residuals = y_train - train_pred

plt.figure()
plt.scatter(X_train["Disp."], train_residuals)
plt.axhline(0, linestyle="--", linewidth=2)
plt.xlabel("Engine displacement")
plt.ylabel("Training residual")
plt.title("Training residuals vs feature")
plt.show()

## 17. Common mistakes to avoid

- **Using the target as an input:** this leaks the answer into the model.
- **Training and testing on the same data:** this does not test generalization.
- **Assuming correlation means causation:** a relationship alone does not prove one variable causes the other.
- **Extrapolating too far:** a line that fits between 1 and 5 litres may be unreliable at 10 litres.
- **Treating R² as accuracy:** R² has a different meaning from classification accuracy.
- **Ignoring units:** always understand what a one-unit increase in X means.
- **Assuming one feature explains everything:** car price also depends on brand, age, mileage, condition, demand, and many other variables.


## 18. Simple Linear Regression vs Multiple Linear Regression

| Simple Linear Regression | Multiple Linear Regression |
|---|---|
| Uses one input feature | Uses two or more input features |
| \(\hat{Y}=mX+c\) | \(\hat{Y}=b_0+b_1X_1+...+b_pX_p\) |
| Easier to visualize in 2D | Harder to visualize beyond a few features |
| Example: price from displacement | Example: price from displacement, age, mileage, brand |

The word “simple” refers to the number of input features, not how easy the problem is.


## 19. R to Python: concept mapping

| Common R workflow | Python equivalent |
|---|---|
| `read.csv()` | `pd.read_csv()` |
| `lm(y ~ x, data=df)` | `LinearRegression().fit(X, y)` |
| `predict(model, newdata=...)` | `model.predict(new_data)` |
| `summary(model)` | Inspect `coef_`, `intercept_`, metrics, and statistical packages as needed |
| `plot(x, y)` | `plt.scatter(x, y)` |

Python's scikit-learn focuses on a consistent fit/predict workflow. Its `LinearRegression` object does not produce the same statistical summary table as R's `summary(lm(...))` by default.


## 20. Practice questions

Try answering these without looking back:

1. What is the difference between a feature and a target?
2. What does a positive slope mean in this car example?
3. If the slope is 0, what does the fitted line look like?
4. What does the intercept represent mathematically?
5. Why do we keep test data separate from training data?
6. What is a residual?
7. Which metric is in the same units as the target: MSE or RMSE?
8. Why does MSE penalize a large error more than a small error?
9. Can R² be negative on test data?
10. Why should we be careful when predicting far outside the training range?

### Mini coding challenge

Change the synthetic data generation so that price has a negative relationship with displacement. Re-run the notebook and observe the slope and scatter plot. Then add a second feature and consider how the problem changes to multiple linear regression.


## 21. Key takeaways

- Simple linear regression predicts a numeric target using one feature.
- The model learns a line: \(\hat{Y}=mX+c\).
- The slope describes the predicted change in Y for a one-unit increase in X.
- The intercept is the model's predicted Y at X = 0.
- Residuals are actual minus predicted values.
- MAE, MSE, and RMSE measure prediction error in different ways.
- R² compares the model against a mean-prediction baseline.
- Always inspect the data, visualize the fit and residuals, and evaluate on unseen data.

**Teaching tip:** Ask learners to predict the direction of the slope from the scatter plot before fitting the model. Then ask them to explain the slope and MAE in everyday language after training.
